# 07j — Targeted near-miss OFFICE audit

Stage 07i showed the frozen OFFICE policy is conservative in coverage. This stage audits only the users added by two one-step relaxations:

- margin: `3 dates / .30 share / .10 margin` → `.05 margin`;
- share: `3 dates / .30 share / .10 margin` → `.20 share`.

No production threshold changes here. The audit compares near-miss users with the frozen 16-user OFFICE cohort using existing Stage-05 reliability tests plus BCL-primary context.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))

def run(cmd,cwd=None): subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)

spec=importlib.util.spec_from_file_location('s07j',REPO_DIR/'analysis'/'07j_near_miss_office_audit.py')
assert spec is not None and spec.loader is not None
s07j=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07j
spec.loader.exec_module(s07j)

print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print(s07j.synthetic_self_check())


In [ ]:
V=Path('/mnt/geolife-data')
C=V/'cache'/'cp2_v2'/'07j_near_miss_office_audit'; C.mkdir(parents=True,exist_ok=True)
C05=V/'cache'/'cp2_v2'/'05_home_office_reliability_validation'
C07I=V/'cache'/'cp2_v2'/'07i_threshold_sensitivity'

paths={
 'office_details':C07I/'office_gate_emitted_details_private.pkl',
 'assignments':C05/'assignments_private.pkl',
 'split':C05/'split_half_details_private.pkl',
 'holdout':C05/'holdout_details_private.pkl',
 'dropout':C05/'dropout_details_private.pkl',
 'shift':C05/'time_shift_details_private.pkl',
 'bcl':C07I/'bcl_anchor_metrics_150m_private.pkl',
}
missing=[str(p) for p in paths.values() if not p.exists()]
if missing: raise FileNotFoundError('Missing upstream artifacts:\n'+'\n'.join(missing))

office_details=pd.read_pickle(paths['office_details'])
assignments=pd.read_pickle(paths['assignments'])
split_details=pd.read_pickle(paths['split'])
holdout_details=pd.read_pickle(paths['holdout'])
dropout_details=pd.read_pickle(paths['dropout'])
shift_details=pd.read_pickle(paths['shift'])
bcl_metrics=pd.read_pickle(paths['bcl'])

cohort=s07j.build_audit_cohort(office_details)
counts=s07j.cohort_counts(cohort)
display(counts)

expected={'baseline':16,'margin_near':2,'share_near':7}
actual=dict(zip(counts['audit_group'].astype(str),counts['users'].astype(int)))
assert actual==expected, f'Expected frozen near-miss counts {expected}, got {actual}'
assert cohort['user_id'].nunique()==25
print('near-miss cohort reproduction: PASS (16 baseline + 2 margin + 7 share = 25 unique)')


In [ ]:
panel=s07j.build_near_miss_audit_panel(
    office_details,
    assignments,
    split_details,
    holdout_details,
    dropout_details,
    shift_details,
    bcl_metrics,
)
assert len(panel)==25
assert panel['candidate_location_id'].notna().all()
print('full-period candidate identity validation: PASS')

behavior_summary=s07j.summarize_behavioral_robustness(panel)
bcl_summary=s07j.summarize_bcl_context(panel)
support_summary=s07j.summarize_support_distribution(panel)
private_signatures=s07j.private_signature_table(panel)


In [ ]:
print('SUPPORT DISTRIBUTION')
display(support_summary)

print('BEHAVIORAL / IDENTITY ROBUSTNESS')
display(behavior_summary)

print('BCL PRIMARY CONTEXT — 100m / 150m')
display(bcl_summary)


In [ ]:
# Additional aggregate diagnostics without exposing user IDs.
near=panel.loc[panel['audit_group'].ne('baseline')].copy()
diagnostics=pd.DataFrame([
    {
        'near_miss_users':int(len(near)),
        'production_home_collision_users':int(near['candidate_equals_production_home'].fillna(False).sum()),
        'both_static_comparators_match_users':int(near['static_comparator_match_count'].eq(2).sum()),
        'both_split_tests_match_users':int(near['split_full_candidate_match_count'].eq(2).sum()),
        'heldout_full_candidate_top1_users':int(near['full_candidate_heldout_top1'].fillna(False).sum()),
        'dropout_retention_ge_75pct_users':int(pd.to_numeric(near['dropout_candidate_retention'],errors='coerce').ge(.75).sum()),
        'bcl_evaluable_users':int(near['bcl_evaluable'].fillna(False).sum()),
        'bcl_work_context_100m_users':int(near['work_compatible_lexical_within_100m'].fillna(False).sum()),
        'bcl_business_name_100m_users':int(near['business_name_within_100m'].fillna(False).sum()),
        'bcl_work_context_150m_users':int(near['work_compatible_lexical_within_150m'].fillna(False).sum()),
        'bcl_business_name_150m_users':int(near['business_name_within_150m'].fillna(False).sum()),
    }
])
print('NEAR-MISS AGGREGATE DIAGNOSTICS')
display(diagnostics)

# Descriptive cross-tabs: no composite score is created.
cross_static=pd.crosstab(near['audit_group'],near['static_comparator_match_count'])
cross_split=pd.crosstab(near['audit_group'],near['split_full_candidate_match_count'])
print('STATIC COMPARATOR MATCH COUNT')
display(cross_static)
print('SPLIT FULL-CANDIDATE MATCH COUNT')
display(cross_split)


In [ ]:
# Private user-level audit panel.
panel.to_pickle(C/'near_miss_office_audit_private.pkl')
private_signatures.to_pickle(C/'near_miss_office_signatures_private.pkl')

# Aggregate outputs only.
counts.to_csv(C/'cohort_counts.csv',index=False)
support_summary.to_csv(C/'support_distribution.csv',index=False)
behavior_summary.to_csv(C/'behavioral_robustness.csv',index=False)
bcl_summary.to_csv(C/'bcl_context_summary.csv',index=False)
diagnostics.to_csv(C/'near_miss_diagnostics.csv',index=False)
cross_static.to_csv(C/'static_comparator_match_distribution.csv')
cross_split.to_csv(C/'split_match_distribution.csv')
print('saved:',C)


## Interpretation boundary

- Near-miss users are defined only by one-step share or margin relaxation at the frozen `min_dates=3` support gate.
- The baseline 16-user OFFICE cohort is the reference, not ground truth.
- Split-half, held-out, dropout, HoWDe and recurrence metrics are source-native reliability diagnostics; they are not combined into a new score.
- The 12-hour shift is a falsification diagnostic, not positive support.
- BCL is supporting historical context only; lack of BCL evidence does not prove a candidate is not an office.
- No near-miss user is promoted automatically by this notebook.
